# 03 · Graph Construction

Builds k-NN similarity graph snapshots from the processed training data and visualizes one of them.

**Prerequisite:** run `02_data_preprocessing.ipynb` (or `python main.py --stage preprocess`) first so that `data/processed/train.csv` exists.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src import config


In [ ]:
import pandas as pd
from src.graph.graph_builder import build_knn_graph, graph_summary
from src.graph.dynamic_graph import generate_graph_snapshots
from src.data.feature_selection import get_selected_features

assert os.path.exists(config.TRAIN_CSV_PATH), "Run preprocessing first (see notebook 02)."
train_df = pd.read_csv(config.TRAIN_CSV_PATH)
feature_columns = get_selected_features()
print(f"Loaded {len(train_df)} rows, {len(feature_columns)} features.")

## Build a single graph snapshot from the first window of rows

In [ ]:
window = train_df.iloc[:200]
features = window[feature_columns].values.astype("float32")
labels = window["Label"].values.astype("int64")

graph = build_knn_graph(features, labels, k=config.K_NEIGHBORS)
print(graph_summary(graph))
print(graph)

## Build several snapshots via sliding windows

In [ ]:
graphs = generate_graph_snapshots(
    train_df,
    feature_columns,
    window_size=200,
    stride=200,
    max_nodes_per_graph=150,
    k_neighbors=config.K_NEIGHBORS,
)
print(f"Built {len(graphs)} graph snapshots.")
for g in graphs[:5]:
    print(" ", graph_summary(g))

## Visualize one graph snapshot

Nodes are colored by their true attack category. This is the actual structure the GNN models will learn to pass messages over.

In [ ]:
from src.utils.visualization import plot_graph_snapshot

fig = plot_graph_snapshot(graphs[0], max_nodes=100, title="Sample dynamic graph snapshot")